In [ ]:
from sklearn.decomposition import PCA
n = 7
xy = torch.stack(torch.meshgrid(*[torch.arange(n/1)]*2), 2).flatten(0, 1)
PCA(1).fit_transform(xy)

In [ ]:
x = (torch.rand(64)+100)*1000
soft_rank(16/x.std()*x[None]).sort()[0].diff().mean()

In [ ]:
64/x.std()*x

In [ ]:
x

In [ ]:
x[x.argsort()][x.argsort()]

In [ ]:
torch.arange(len(x))[S]

In [ ]:
s = x.argsort()
S = torch.empty_like(s)
S[s] = torch.arange(len(s))
s, S

In [ ]:
x

In [ ]:
torch.Tensor.argsort

In [ ]:
import torch
from torchsort import soft_rank
import matplotlib.pyplot as plt
n = 128
xy = torch.stack(torch.meshgrid(*[torch.arange(n/1)]*2),2).flatten(0, 1)
w = torch.pdist(xy).pow_(-2)
w /= w.sum()
L = 9e9
for _ in range(1024):
    x = torch.rand(n*n, requires_grad=True)
    o = torch.optim.Adam([x], 1)
    O = torch.optim.lr_scheduler.CosineAnnealingLR(o, 1000, .1)
    l_,c = 9e9,0
    for _ in range(1000):
        l = torch.pdist(soft_rank(4/x.std()*x[None]).T).square()@w
        l.backward()
        o.step()
        o.zero_grad()
        O.step()
        if l < l_:
            l_ = l.data
            c = 0
        elif c < 4:
            c += 1
        else:
            break
    x = torch.empty(n*n).put_(x.argsort(),torch.arange(n*n/1))
    l = torch.pdist(x[:,None]).square_()@w
    if l < L:
        X = x
        L = l
        plt.subplots_adjust(0,0,1,1,0,0)
        plt.plot(*xy[x.long()].T+torch.linspace(0,.5,n*n))
        clear_output(True)
        plt.show()

In [ ]:
plt.plot(xy[X.long(),0])

In [ ]:
plt.plot(xy[X.long()[:200],1])

In [ ]:
import numpy as np
import torch
from torch.nn import *
from IPython.display import clear_output
from torchsort import soft_rank
import matplotlib.pyplot as plt
n = 8
xy = torch.stack(torch.meshgrid(*[torch.arange(n/1)]*2),2).flatten(0, 1)
w = torch.pdist(xy).pow_(-2)
w /= w.sum()
L = 9e9
for _ in range(1):
    x = torch.rand(n,n).add_(torch.arange(n/1)[:,None]).flatten().requires_grad_()
    o = torch.optim.Adam([x], 1)
    O = torch.optim.lr_scheduler.CosineAnnealingLR(o, 10000, .1)
    for t in range(1,10000):
        l = torch.pdist(x[:,None])
        l,l2 = l.square()@w, (-l).exp().mean()#(1-l).relu_().mean()
        (l+(i/10000)*l2).backward()
        o.step()
        o.zero_grad()
        O.step()
        if not t%1000:
            print(l, l2)
    x = x.argsort()
    l = torch.pdist(x.float()[:,None]).square()@w
    print(l)
    if l < L:
        X = x
        L = l
        plt.subplots_adjust(0,0,1,1,0,0)
        plt.plot(*xy[x].T+torch.linspace(0,1,n*n))
        # clear_output(True)
        plt.show()

In [ ]:
np.log1p(np.exp(1))

In [ ]:
from ortools.sat.python.cp_model import CpModel, CpSolver
import matplotlib.pyplot as plt
def New(l,u,_):
    return m.NewIntVar(l,u,_)
def Abs(x,l,u):
    return m.AddAbsEquality(_:=m.NewIntVar(l,u,f'abs{x.name}'),x)and _
def Sub(x,y,l,u):
    return m.Add((_:=m.NewIntVar(l,u,f'{x.name}-{y.name}'))==x)and _
def Sq(x,l,u):
    return m.AddMultiplicationEquality(_:=m.NewIntVar(l,u,f'sq{x.name}'),x,x)and _
a,b = 8,8
m = CpModel()
X = [(New(0,a*b-1,f'{i}{j}'),i,j)for i in range(a)for j in range(b)]
m.AddAllDifferent([x for x,i,j in X])
m.Minimize(_:=sum(1/((i-k)**2+(j-l)**2)*Sq(Sub(x,y,1-a*b,a*b-1),1,(a*b-1)**2)for _,(x,i,j)in enumerate(X)for y,k,l in X[:_]))
o = CpSolver()
o.parameters.max_time_in_seconds = 3600
o.parameters.num_search_workers = 12
o.Solve(m)
o = o.Values([x for x,i,j in X]).to_numpy()
plt.plot(o//a, o%a)